In [1]:
from newsapi import NewsApiClient
import yfinance as yf

from Utils.clean_data_helper import *

'''
If unable to install pygooglenews, try
- pip install "setuptools<58.0"
- pip install feedparser --upgrade
- pip install setuptools
- pip install pygooglenews --upgrade
'''
from pygooglenews import GoogleNews
from tqdm import tqdm
import requests
import json


# Get S&P500 companies' tickers

In [2]:
response = requests.get('https://stockanalysis.com/list/sp-500-stocks/')
companies_info_df = pd.read_html(response.content)[0]
companies_info_df = companies_info_df.drop(columns=["No.", "Stock Price", "% Change", "Revenue"])
companies_info_df.to_excel("Data/SPY_companies_info.xlsx", index=False)

In [3]:
# tech stocks, pharma stocks, oil stocks, tobacco stocks and Market
tickers = companies_info_df["Symbol"].to_list()

# News Data Download
- newsapi has a limit of 2024-11-01 onwards
- pygoognews https://github.com/kotartemiy/pygooglenews
- Some other packages to try out: https://www.newscatcherapi.com/blog/python-web-scraping-libraries-to-mine-news-data

In [ ]:
gn = GoogleNews(lang = 'en')

news_dfs = []
for ticker in tqdm(tickers):
    top = gn.search(ticker)
    entries = top["entries"]
    df_temp = clean_goog_news(entries)
    news_dfs += [df_temp.copy()]


  3%|▎         | 13/503 [05:49<3:38:58, 26.81s/it]

In [ ]:
news_dfs[0]

184

## General News 
- Newscatcher for general news not targetted at specific stocks: https://github.com/kotartemiy/newscatcher


In [10]:
# topics supported: 'tech', 'news', 'business', 'science', 'finance', 'food', 'politics', 'economics', 'travel', 'entertainment', 'music', 'sport', 'world'

finance_news = clean_newscatcher_news('finance')
business_news = clean_newscatcher_news('business')
economics_news = clean_newscatcher_news('economics')
tech_news = clean_newscatcher_news('tech')
politics_news = clean_newscatcher_news('politics')


No. of URLs: 80
['marketwatch.com', 'investopedia.com', 'seekingalpha.com', 'xe.com', 'thestreet.com', 'financialpost.com', 'investing.com', 'kiplinger.com', 'benzinga.com', 'thisismoney.co.uk', 'cityam.com', 'daveramsey.com', 'fin24.com', 'smartasset.com', 'investorplace.com', 'institutionalinvestor.com', 'ritholtz.com', 'marketrealist.com', 'pionline.com', 'efinancialcareers.com', 'marketoracle.co.uk', 'moneymorning.com', 'financial-planning.com', 'citywire.co.uk', 'wealthmanagement.com', 'investmentwatchblog.com', 'moneytalksnews.com', 'morningstar.co.uk', 'insidermonkey.com', 'risk.net', 'realclearmarkets.com', 'financialsamurai.com', 'worldfinance.com', 'armstrongeconomics.com', 'savingadvice.com', 'fool.co.uk', 'elliottwave.com', 'simplywall.st', 'kitces.com', 'etfdailynews.com', 'investmentweek.co.uk', 'globalcapital.com', 'interest.co.nz', 'ai-cio.com', 'fool.ca', 'learnbonds.com', 'wallstreetdaily.com', 'abladvisor.com', 'fool.com.au', 'investmentexecutive.com', 'investmentu.c

In [12]:
all_news = pd.concat([finance_news, business_news, economics_news, tech_news, politics_news])
all_news

,date,title,source,topic
0,"Mon, 02 Dec 2024 12:08:00 GMT","Gold, silver under pressure as dollar climbs",https://www.marketwatch.com/story/gold-silver-...,finance
1,"Mon, 02 Dec 2024 11:54:00 GMT",All of Wall Street expects stocks to rise — an...,https://www.marketwatch.com/story/all-of-wall-...,finance
2,"Mon, 02 Dec 2024 11:49:00 GMT",BlackRock near deal to buy private credit mana...,https://www.marketwatch.com/story/blackrock-ne...,finance
3,"Mon, 02 Dec 2024 11:27:00 GMT",‘My house is paid off’: How do I ensure my son...,https://www.marketwatch.com/story/ive-accumula...,finance
4,"Mon, 02 Dec 2024 11:22:00 GMT",My friend declared bankruptcy. Can creditors s...,https://www.marketwatch.com/story/my-friend-de...,finance
...,...,...,...,...
1316,2024-03-03T05:47:23Z,The world of bullshit we’ve built: Reflections...,https://clubtroppo.com.au/2024/03/03/the-stran...,politics
1317,2024-02-13T04:48:17Z,Figuring out the strange new rules of resource...,https://clubtroppo.com.au/2024/02/13/figuring-...,politics
1318,2023-12-27T08:28:06Z,William Hague gets on board,https://clubtroppo.com.au/2023/12/27/william-h...,politics
1319,2023-12-23T05:49:25Z,Michael Polanyi in 1960 on Teilhard de Chardin...,https://clubtroppo.com.au/2023/12/23/michael-p...,politics


# Stocks Data Download
- Note that this is not the returns of the stocks data
- Let's focus on periods 2023-01-01 to 2024-12-01 for this project


In [7]:
stocks = yf.download(tickers, threads=True, group_by='ticker', start='2023-01-01', end='2024-12-01')

[*********************100%***********************]  503 of 503 completed

2 Failed downloads:
['BF.B']: YFPricesMissingError('$%ticker%: possibly delisted; no price data found  (1d 2023-01-01 -> 2024-12-01)')
['BRK.B']: YFTzMissingError('$%ticker%: possibly delisted; no timezone found')


In [ ]:
stocks

# Save Data

In [ ]:
with pd.ExcelWriter('Data/news_data.xlsx') as writer:
    for i, news_df in enumerate(news_dfs):
        news_df.to_excel(writer, sheet_name=tickers[i], index=False)
    all_news.to_excel(writer, sheet_name="General", index=False)
# Close the ExcelWriter object
# writer.save()

c:\Users\Ethelda\anaconda3\lib\site-packages\xlsxwriter\workbook.py:336: UserWarning: Calling close() on already closed file.
  warn("Calling close() on already closed file.")


In [ ]:
stocks.to_excel("Data/stocks_data.xlsx")